In [6]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.retrievers import EnsembleRetriever, BM25Retriever
from langchain_core.documents import Document


In [7]:
#Step1 : Sample Documents
docs = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for semantic search."),
    Document(page_content="The Eiffel Tower is located in Paris."),
    Document(page_content="Langchain can be used to develop agentic ai application."),
    Document(page_content="Langchain has many types of retrievers.")
]

#Step 2 : Dense Retriever (FAISS + Huggingface)
embedding_model = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
dense_vectorstore = FAISS.from_documents(docs,embedding_model)
dense_retriever = dense_vectorstore.as_retriever()




Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11228.71it/s]


In [8]:
##Sparse Retriever(BM25)

sparse_retriever = BM25Retriever.from_documents(docs)
sparse_retriever.k=3 #top k documents to retrieve

## Step 4 : Combine with Ensemble Retriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.7,0.3]
)

In [9]:
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x12bb76fd0>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x12b9402f0>, k=3)], weights=[0.7, 0.3])

In [10]:
#Step 5 : Query and get results
query = "How can I build an application using LLMs?"
results = hybrid_retriever.invoke(query)

#step 6 : Print Results:
for i, doc in enumerate(results):
    print(f"\n Document {i+1} : \n {doc.page_content}")


 Document 1 : 
 LangChain helps build LLM applications.

 Document 2 : 
 Langchain can be used to develop agentic ai application.

 Document 3 : 
 Langchain has many types of retrievers.

 Document 4 : 
 Pinecone is a vector database for semantic search.


## RAG pipeline with hybrid retriever

In [11]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain


In [14]:
#step 5 : Prompt Template
prompt = PromptTemplate.from_template("""
Answer the question based on the context below:
Context:
{context}

Question:
{input}
""")

## Step 6 : llm
llm = init_chat_model("groq:openai/gpt-oss-20b", temperature = 0.2)
llm


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.10', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x134d39550>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x134d39fd0>, model_name='openai/gpt-oss-20b', temperature=0.2, model_kwargs={}, groq_api_key=SecretStr('**********'))

In [15]:
### Create Stuff Document Chain
document_chain = create_stuff_documents_chain(llm=llm,prompt=prompt)

## create full rag chain
rag_chain = create_retrieval_chain(retriever=hybrid_retriever, combine_docs_chain= document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x12bb76fd0>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x12b9402f0>, k=3)], weights=[0.7, 0.3]), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based on the context below:\nContext:\n{context}\n\nQuestion:\n{input}\n')
            | ChatGroq(metadata={'lc_versions':

In [18]:
##Step 9 : Ask a question
query = {"input":"How can I build an app using LLMs?"}
response = rag_chain.invoke(query)

## step 10 : Output
print("Answere: \n, ",response["answer"])
print("\n source document:")
for i , doc in enumerate(response["context"]):
    print(f"\nDoc {i+1} : {doc.page_content}")

Answere: 
,  **Building an LLM‑powered app with LangChain (and optional Pinecone)**  

Below is a practical, step‑by‑step recipe that covers the most common use cases: a simple chatbot, a knowledge‑base‑driven agent, and a semantic‑search app.  
Feel free to skip or reorder steps depending on your project’s scope.

| Step | What to do | Why it matters | Quick code snippet |
|------|------------|----------------|--------------------|
| 1️⃣ | **Define the problem** | Clarify the user goal (e.g., FAQ bot, document summarizer, recommendation engine). | – |
| 2️⃣ | **Pick an LLM** | Choose a model that fits your latency, cost, and privacy needs (OpenAI GPT‑4, Anthropic Claude, Llama‑2, etc.). | `from langchain.chat_models import ChatOpenAI` |
| 3️⃣ | **Set up LangChain** | Install LangChain (`pip install langchain`) and import the relevant modules. | `from langchain import PromptTemplate, LLMChain` |
| 4️⃣ | **Create a prompt** | Design a prompt template that guides the LLM. | `prompt = Pro